# 🏗️ Building & Training a GPT-2-like LLM from Scratch

## 📚 Tutorial Overview

This notebook provides a **hands-on, educational walkthrough** of training a GPT-2-style language model **from scratch** — meaning the model weights are **randomly initialized**, not downloaded from a pretrained checkpoint. The only pretrained component we borrow is the **GPT-2 BPE tokenizer** (vocabulary and merge rules), since training a tokenizer from scratch is a separate topic.

> **🔑 Key distinction:** This is **native GPT-2 training** — we build and train the full transformer from random weights. This is NOT fine-tuning or LoRA on top of a pretrained model. Think of it as re-running GPT-2's original training recipe on a small scale.

The tutorial is structured in two complementary parts:

| Part | Approach | Goal |
|------|----------|------|
| **Part 1** | Use Hugging Face `GPT2LMHeadModel` (random init) | Quickly train a small GPT-2 using the high-level Trainer API with train/val benchmarks |
| **Part 2** | Implement GPT-2 from scratch in PyTorch | Understand every component: attention, MLP, transformer blocks, embeddings |

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Load and preprocess a text dataset for causal language modeling with **proper train/val/test splits**
- Configure a small GPT-2 model (~18M–70M parameters) with randomly initialized weights
- Train using Hugging Face's `Trainer` API with mixed precision and validation benchmarks
- Implement **Multi-Head Causal Self-Attention** from scratch in PyTorch
- Build the full GPT-2 architecture: embeddings → transformer blocks → LM head
- Train the custom model with a manual training loop (15 epochs) and per-epoch validation
- Diagnose and fix mode collapse (label imbalance) by tuning model capacity
- Generate text and evaluate with loss/perplexity on a held-out test set

## 📊 Dataset & Splits

We use the [Twitter Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment) dataset from Kaggle — a sentiment-classification dataset with ~14K tweets labeled as *positive*, *negative*, or *neutral*. We format each example as an instruction-following prompt for causal LM training.

| Split | Size | Purpose |
|-------|------|---------|
| **Train** | 5,000 | Model weight updates |
| **Validation** | 1,000 | Hyperparameter tuning & early stopping |
| **Test** | 1,000 | Final held-out evaluation |

> **⚠️ Important:** This dataset is for **sentiment analysis**, not general-purpose LLM pretraining. The goal is to learn the training workflow and architecture — not to build a chatbot.

## 🖥️ Environment

- **GPU:** NVIDIA GeForce RTX 5090 Laptop (24 GB VRAM)
- **Framework:** PyTorch 2.13 + Transformers 5.x + CUDA 13.2
- **Conda environment:** `agentic_ai` (Python 3.13)


## 1. Load, Split, and Preprocess the Dataset

In this section we:
1. **Load** the CSV dataset with pandas and take the first 7,000 rows
2. **Shuffle** deterministically (seed=42) and split into **train (5,000) / validation (1,000) / test (1,000)**
3. **Define an instruction template** that wraps each tweet in a prompt format:
   ```
   Instruction: <task description>
   Input: <tweet>
   Output: <sentiment label>
   ```
4. **Compute token budgets** — we need all examples to fit within a fixed `max_total_tokens`, so we calculate how many tokens are consumed by the fixed template and reserve the rest for the tweet + label
5. **Truncate tweets** that exceed the budget using the GPT-2 tokenizer

> **Why fixed-length sequences?** GPT-2 uses learned positional embeddings with a fixed maximum length (`n_positions`). All training sequences must be ≤ this limit, and we pad shorter sequences to this length for efficient batching.

> **🔑 Why train/val/test splits?** The validation set lets us monitor overfitting during training; the test set provides an unbiased final evaluation on never-seen data.

### ⬇️ Step 0: Download GPT-2 Tokenizer Locally

We pre-download only the **tokenizer files** (~2 MB) from ModelScope to avoid hf-mirror.com timeouts. Since we only need `GPT2Tokenizer`, not the full 700 MB model, this step downloads just the vocabulary, merge rules, and tokenizer config.

In [1]:
# -------------------------------------------------------------------
# 0. Pre-download GPT-2 tokenizer files only (~2 MB) from ModelScope.
#    We only need the tokenizer, not the full model — saved to ./models/gpt2-tokenizer/
# -------------------------------------------------------------------
import os

LOCAL_GPT2_DIR = "./models/gpt2-tokenizer"  # Small (~2 MB), kept in tutorials folder

if not os.path.exists(LOCAL_GPT2_DIR):
    print(f"Downloading GPT-2 tokenizer from ModelScope to: {LOCAL_GPT2_DIR}")
    from modelscope import snapshot_download
    snapshot_download(
        "AI-ModelScope/gpt2",                    # GPT-2 base on ModelScope
        local_dir=LOCAL_GPT2_DIR,
        allow_file_pattern=[                     # Tokenizer files ONLY (~2 MB total):
            "tokenizer_config.json",             #   - tokenizer configuration
            "tokenizer.json",                    #   - fast tokenizer data
            "vocab.json",                        #   - BPE vocabulary (50,257 tokens)
            "merges.txt",                        #   - BPE merge rules
            "special_tokens_map.json",           #   - special token definitions
        ],
    )
    print("✓ Tokenizer download complete.")
else:
    print(f"✓ GPT-2 tokenizer already cached at: {LOCAL_GPT2_DIR}")

✓ GPT-2 tokenizer already cached at: ./models/gpt2-tokenizer


In [2]:
import pandas as pd
import torch
import random
from transformers import GPT2Tokenizer, GPT2LMHeadModel, GPT2Config, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset

# -------------------------------------------------------------------
# 0. Load tokenizer early — we need it to measure token lengths
# -------------------------------------------------------------------
tokenizer = GPT2Tokenizer.from_pretrained("./models/gpt2-tokenizer")  # Local tokenizer (~2 MB, Step 0)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token; reuse EOS

# -------------------------------------------------------------------
# 1. Load the dataset and split into train / validation / test
# -------------------------------------------------------------------
df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv')

# Take 7000 rows: 5000 train + 1000 val + 1000 test
df = df.head(7000).copy()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

df_train = df.iloc[:5000]
df_val   = df.iloc[5000:6000]
df_test  = df.iloc[6000:7000]

print(f"Dataset splits — Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# -------------------------------------------------------------------
# 2. Define the instruction template and compute token budget
# -------------------------------------------------------------------
instruction = "Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:"

# The fixed part of every example (without the tweet content or label)
fixed_template = f"Instruction: {instruction}\nInput:\nOutput:"
fixed_token_count = len(tokenizer.encode(fixed_template))
print(f"Fixed tokens (without tweet and label): {fixed_token_count}")

# Total tokens per example = fixed_template + tweet + 1 (label token)
# 31 (fixed) + 80 (tweet budget) = 111 total tokens per sequence
max_total_tokens = 31 + 80  # = 111 — must match n_positions in GPT2Config later!
max_tweet_tokens = max_total_tokens - fixed_token_count - 1  # reserve 1 token for label
print(f"Max tokens allowed for tweet content: {max_tweet_tokens}")

# -------------------------------------------------------------------
# 3. Helper: truncate a tweet to fit within the token budget
# -------------------------------------------------------------------
def truncate_tweet(tweet, max_tokens):
    """Truncate tweet text to at most `max_tokens` GPT-2 tokens."""
    tokens = tokenizer.encode(tweet, truncation=True, max_length=max_tokens)
    return tokenizer.decode(tokens, skip_special_tokens=True)

# -------------------------------------------------------------------
# 4. Build formatted examples for each split
# -------------------------------------------------------------------
def build_dataset(df_split):
    """Convert a DataFrame split into a Hugging Face Dataset of formatted prompts."""
    texts = []
    for _, row in df_split.iterrows():
        short_tweet = truncate_tweet(row['content'], max_tweet_tokens)
        text = f"Instruction: {instruction}\nInput: {short_tweet}\nOutput: {row['sentiment']}"
        texts.append(text)
    return Dataset.from_dict({"text": texts})

dataset_train = build_dataset(df_train)
dataset_val   = build_dataset(df_val)
dataset_test  = build_dataset(df_test)

print(f"Created datasets — Train: {len(dataset_train)} | Val: {len(dataset_val)} | Test: {len(dataset_test)}")

c:\Users\ycasi\anaconda3\envs\agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0804 16:04:02.855000 23408 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Dataset splits — Train: 5000 | Val: 1000 | Test: 1000
Fixed tokens (without tweet and label): 31
Max tokens allowed for tweet content: 79
Created datasets — Train: 5000 | Val: 1000 | Test: 1000


> **🔑 Key constraint:** `max_total_tokens` must equal `n_positions` in the GPT-2 configuration. The model's positional embedding matrix has shape `(n_positions, n_embd)`, so every input must be exactly `n_positions` tokens long (shorter sequences are padded).

## 2. Tokenize All Splits

Now we convert each text example into token IDs using the GPT-2 Byte-Pair Encoding (BPE) tokenizer. We apply the same tokenization to **all three splits** (train, validation, and test):

- **Truncation**: sequences longer than `max_total_tokens` are cut off
- **Padding**: shorter sequences are padded to `max_total_tokens` using the EOS token
- **No `return_tensors`**: we let the `DataCollatorForLanguageModeling` handle batching later

The `.map()` call applies tokenization to each dataset in parallel (batched mode).

In [3]:
# -------------------------------------------------------------------
# 2. Tokenize all three splits
# -------------------------------------------------------------------
def tokenize_function(examples):
    """
    Convert text examples to token IDs.
    - Truncates to max_total_tokens if the text is too long
    - Pads to max_total_tokens if the text is shorter
    - Does NOT return tensors — the data collator will batch and tensorize later
    """
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_total_tokens,     # must match n_positions in GPT2Config
    )

# Apply tokenization to each split (batched for speed)
tokenized_train = dataset_train.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val   = dataset_val.map(tokenize_function,   batched=True, remove_columns=["text"])
tokenized_test  = dataset_test.map(tokenize_function,  batched=True, remove_columns=["text"])

print(f"Tokenized — Train: {len(tokenized_train)} | Val: {len(tokenized_val)} | Test: {len(tokenized_test)}")

Map: 100%|██████████| 1000/1000 [00:00<00:00, 23162.07 examples/s]

Tokenized — Train: 5000 | Val: 1000 | Test: 1000


### 🔬 Diagnosis: Label Distribution & Token Analysis

Before training, let's understand the data imbalance — if one label dominates, the model will collapse to it.

In [4]:
# -------------------------------------------------------------------
# Label distribution check
# -------------------------------------------------------------------
from collections import Counter

for name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    counts = Counter(split_df['sentiment'])
    total = len(split_df)
    print(f"\n{name} label distribution ({total} examples):")
    for label, cnt in counts.most_common():
        bar = "█" * int(cnt / total * 40)
        print(f"  {label:<12} {cnt:>5} ({cnt/total*100:5.1f}%)  {bar}")

# What does the model actually need to predict?
print("\n--- Token budget breakdown ---")
print(f"  Fixed template tokens:  {fixed_token_count}")
print(f"  Max tweet tokens:       {max_tweet_tokens}")
print(f"  Label (1 token):        1")
print(f"  Total:                  {max_total_tokens}")
print(f"  Padding:                {max_total_tokens - fixed_token_count - 1 - 52.4:.0f} avg")
print(f"\n  ⚠️  The label is only 1 token out of ~52 real tokens (~2% of content)")
print(f"  ⚠️  The model spends 98% of its capacity predicting template words & padding")
print(f"  ⚠️  If 'worry' is the most frequent label, a small model defaults to it")

# Check what token "worry" maps to
worry_id = tokenizer.encode(" worry")[0]  # include leading space (BPE convention)
print(f"\n  Token ID for ' worry': {worry_id}")
print(f"  Token ID for ' sadness': {tokenizer.encode(' sadness')[0]}")
print(f"  Token ID for ' neutral': {tokenizer.encode(' neutral')[0]}")
print(f"  Token ID for ' positive': {tokenizer.encode(' positive')[0]}")


Train label distribution (5000 examples):
  worry         1532 ( 30.6%)  ████████████
  sadness       1122 ( 22.4%)  ████████
  neutral        969 ( 19.4%)  ███████
  hate           283 (  5.7%)  ██
  surprise       278 (  5.6%)  ██
  happiness      216 (  4.3%)  █
  love           178 (  3.6%)  █
  fun            108 (  2.2%)  
  relief          99 (  2.0%)  
  empty           98 (  2.0%)  
  enthusiasm      59 (  1.2%)  
  boredom         35 (  0.7%)  
  anger           23 (  0.5%)  

Val label distribution (1000 examples):
  worry          334 ( 33.4%)  █████████████
  sadness        210 ( 21.0%)  ████████
  neutral        185 ( 18.5%)  ███████
  surprise        64 (  6.4%)  ██
  hate            48 (  4.8%)  █
  happiness       45 (  4.5%)  █
  love            33 (  3.3%)  █
  fun             21 (  2.1%)  
  relief          20 (  2.0%)  
  empty           16 (  1.6%)  
  enthusiasm      12 (  1.2%)  
  boredom          8 (  0.8%)  
  anger            4 (  0.4%)  

Test label distri

## 3. Inspect the Tokenized Data (Train Split)

Always sanity-check your tokenized data before training! We'll inspect examples from the **train split** to verify:
- The instruction/input/output format is preserved after tokenization → decoding
- Padding tokens are correctly applied (tokens equal to `pad_token_id`)
- Token-length statistics across the dataset

In [5]:
# -------------------------------------------------------------------
# 3. Inspect the tokenized dataset (train split) — with padding visualization
# -------------------------------------------------------------------
print("\n--- Inspection of tokenized train dataset ---")
print(f"PAD token ID: {tokenizer.pad_token_id}  (= EOS token)")

for i in range(3):
    sample = tokenized_train[i]
    input_ids = sample["input_ids"]
    attention_mask = sample["attention_mask"]

    # Count real vs pad tokens
    real_tokens = sum(attention_mask)
    pad_tokens = len(attention_mask) - real_tokens

    print(f"\n{'='*70}")
    print(f"Sample {i}:")
    print(f"  Total length: {len(input_ids)}  |  Real tokens: {real_tokens}  |  Padding tokens: {pad_tokens}")
    print(f"  First 10 and last 10 token IDs (PAD={tokenizer.pad_token_id}):")
    print(f"    ...{input_ids[:10]}... ...{input_ids[-10:]}...")
    print(f"  Attention mask (first 10, last 10):")
    print(f"    ...{attention_mask[:10]}... ...{attention_mask[-10:]}...")
    print(f"  Decoded (excluding padding):")
    decoded = tokenizer.decode(input_ids, skip_special_tokens=True)
    print(f"    {decoded[:120]}...")
    print(f"  Decoded length (chars): {len(decoded)}")

# Overall statistics
token_lengths = [sum(s["attention_mask"]) for s in tokenized_train]
print(f"\n--- Train split token statistics ---")
print(f"  Real tokens — Max: {max(token_lengths)} | Min: {min(token_lengths)} | Avg: {sum(token_lengths)/len(token_lengths):.1f}")
print(f"  Padding rate: {1 - sum(token_lengths)/(len(token_lengths)*111):.1%}")


--- Inspection of tokenized train dataset ---
PAD token ID: 50256  (= EOS token)

Sample 0:
  Total length: 111  |  Real tokens: 51  |  Padding tokens: 60
  First 10 and last 10 token IDs (PAD=50256):
    ...[6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708]... ...[50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]...
  Attention mask (first 10, last 10):
    ...[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]... ...[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...
  Decoded (excluding padding):
    Instruction: Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:
I...
  Decoded length (chars): 214

Sample 1:
  Total length: 111  |  Real tokens: 44  |  Padding tokens: 67
  First 10 and last 10 token IDs (PAD=50256):
    ...[6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708]... ...[50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]...
  Attention mask (first 10, last 10):
    ...[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]... ...[

### 🔍 Dataloader Batch Inspection

Let's sanity-check what `DataCollatorForLanguageModeling` produces: shapes, label alignment, and attention mask correctness.

In [6]:
from torch.utils.data import DataLoader

# Temporary data collator + dataloader to inspect
_temp_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
_temp_loader = DataLoader(tokenized_train, batch_size=4, shuffle=False, collate_fn=_temp_collator)

for batch_idx, batch in enumerate(_temp_loader):
    if batch_idx >= 2:  # Inspect only first 2 batches
        break
    input_ids = batch["input_ids"]
    labels = batch["labels"]
    attention_mask = batch["attention_mask"]
    B, T = input_ids.shape
    print(f"\n{'='*60}")
    print(f"Batch {batch_idx}: input_ids shape={list(input_ids.shape)}, labels shape={list(labels.shape)}")
    print(f"  Attention mask per sample: {attention_mask.sum(dim=1).tolist()} real tokens out of {T}")
    print(f"  Label alignment check (input vs labels, first 15 tokens of sample 0):")
    print(f"    Input:  {input_ids[0,:15].tolist()}")
    print(f"    Labels: {labels[0,:15].tolist()}")
    print(f"    (labels should match input for causal LM — shifting happens in model loss)")
    # Show decoded sample
    decoded_sample = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    print(f"  Decoded sample 0 (first 100 chars): {decoded_sample[:100]}...")


Batch 0: input_ids shape=[4, 111], labels shape=[4, 111]
  Attention mask per sample: [51, 44, 70, 52] real tokens out of 111
  Label alignment check (input vs labels, first 15 tokens of sample 0):
    Input:  [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13, 25235, 3446, 530]
    Labels: [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13, 25235, 3446, 530]
    (labels should match input for causal LM — shifting happens in model loss)
  Decoded sample 0 (first 100 chars): Instruction: Analyze the sentiment of the following tweet. Output exactly one word, with no punctuat...

Batch 1: input_ids shape=[4, 111], labels shape=[4, 111]
  Attention mask per sample: [65, 63, 44, 80] real tokens out of 111
  Label alignment check (input vs labels, first 15 tokens of sample 0):
    Input:  [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13, 25235, 3446, 530]
    Labels: [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13, 25235, 

---
## 🔵 Part 1 — Training with Hugging Face `GPT2LMHeadModel`

In Part 1 we use the **pre-built GPT-2 implementation** from the `transformers` library. This is the quickest way to get a working model — we just configure the architecture, load the pretrained tokenizer, and let the `Trainer` API handle the training loop, mixed precision, checkpointing, etc.

This approach is ideal for prototyping and understanding the **high-level workflow** before diving into the low-level implementation in Part 2.

---

## 4. Configure a Small GPT-2 Model (~70M Parameters)

We define a compact GPT-2 architecture using `GPT2Config`. **The model weights are randomly initialized** — this is native training, not fine-tuning.

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `vocab_size` | 50,257 | GPT-2's original BPE vocabulary size |
| `n_positions` | 111 | Maximum sequence length (positional embeddings) |
| `n_embd` | 512 | Hidden/embedding dimension — determines model capacity |
| `n_layer` | 6 | Number of transformer (decoder) blocks |
| `n_head` | 8 | Number of attention heads per block (`n_embd` must be divisible by this) |
| `resid_pdrop` | 0.1 | Dropout after residual connections |
| `embd_pdrop` | 0.1 | Dropout on embeddings |
| `attn_pdrop` | 0.1 | Dropout on attention weights |

> **📐 Parameter count:** With `n_embd=512` and `n_layer=6`, this model has ~**70M** parameters. Full GPT-2 Small (124M) uses `n_embd=768, n_layer=12`.

In [7]:
# -------------------------------------------------------------------
# 4. Configure a small GPT-2 model (~110M parameters)
# -------------------------------------------------------------------
config = GPT2Config(
    vocab_size=50257,           # GPT-2 BPE vocabulary size
    n_positions=max_total_tokens,  # Max sequence length (= 111). The positional embedding
                                 # matrix is (n_positions, n_embd) — it maps each position
                                 # to a vector in the embedding space.
    n_embd=512,                 # Embedding dimension (hidden size).
                                # Full GPT-2 Small uses 768; we use 512 for faster training.
    n_layer=6,                  # Number of transformer decoder blocks.
                                # GPT-2 Small uses 12; we halve it.
    n_head=8,                   # Number of attention heads.
                                # n_embd (512) must be divisible by n_head (8) → head_dim = 64.
    resid_pdrop=0.1,            # Dropout probability for residual connections
    embd_pdrop=0.1,             # Dropout probability for embeddings
    attn_pdrop=0.1,             # Dropout probability for attention weights
)

# Instantiate the model with random weights (no pretrained checkpoint)
model = GPT2LMHeadModel(config)
print(f"Model has {model.num_parameters():,} parameters")

Model has 44,703,744 parameters


## 5. Model Architecture Inspection — Layer-by-Layer Breakdown

Before training, let's examine every module in the model: its **type**, **parameter count**, **trainable status**, and **tensor dimensions**.

This helps verify:
- The architecture matches our `GPT2Config`
- No layer is accidentally frozen (all should be trainable)
- The parameter distribution across components (embeddings vs. attention vs. MLP vs. LM head)

In [8]:
# -------------------------------------------------------------------
# 5. Layer-by-layer architecture inspection
# -------------------------------------------------------------------
import torch.nn as nn

def inspect_model(model, indent=0):
    """
    Recursively print every submodule of the model with:
      - Module type (class name)
      - Parameter shapes (if leaf module)
      - Trainable / total parameter count
      - Memory footprint (fp32)
    """
    prefix = "  " * indent
    total_params = 0
    total_trainable = 0

    for name, child in model.named_children():
        # Count parameters for this child
        child_total = sum(p.numel() for p in child.parameters())
        child_trainable = sum(p.numel() for p in child.parameters() if p.requires_grad)

        # Get shape info for leaf modules (Linear, Embedding, LayerNorm, Dropout)
        shape_info = ""
        if isinstance(child, (nn.Linear, nn.Embedding)):
            w = next(child.parameters())
            shape_info = f"  ← weight: {list(w.shape)}"
        elif isinstance(child, nn.LayerNorm):
            w = next(child.parameters())
            shape_info = f"  ← weight: {list(w.shape)}"

        # Memory: 4 bytes per fp32 param (2 for fp16, but stored as fp32)
        mem_mb = child_total * 4 / (1024 ** 2)

        # Print this level
        bar = "│  " if indent > 0 else ""
        print(f"{prefix}{bar}├─ {name:<25} ({type(child).__name__:<18})  "
              f"params: {child_trainable:>10,} / {child_total:>10,}  "
              f"({mem_mb:>6.2f} MB){shape_info}")

        total_params += child_total
        total_trainable += child_trainable

        # Recurse into children (e.g., transformer.h.0, transformer.h.1, ...)
        if list(child.children()):
            inspect_model(child, indent + 1)

    return total_params, total_trainable


print("=" * 90)
print("GPT-2-like Model Architecture — Layer-by-Layer Breakdown")
print("=" * 90)
print()

total_params, total_trainable = inspect_model(model)

print()
print("─" * 90)
print(f"{'TOTAL':>39}  params: {total_trainable:>10,} / {total_params:>10,}  "
      f"({total_params * 4 / (1024**2):>6.2f} MB fp32)")
print()

# -------------------------------------------------------------------
# Parameter breakdown by component type
# -------------------------------------------------------------------
emb_params = sum(p.numel() for name, p in model.named_parameters() if 'wte' in name or 'wpe' in name)
attn_params = sum(p.numel() for name, p in model.named_parameters() if 'attn' in name or 'c_attn' in name or 'c_proj' in name.split('.')[-2:])
mlp_params = sum(p.numel() for name, p in model.named_parameters() if 'mlp' in name)
lm_head_params = sum(p.numel() for name, p in model.named_parameters() if 'lm_head' in name)
ln_params = sum(p.numel() for name, p in model.named_parameters() if 'ln_' in name)
other_params = total_params - emb_params - attn_params - mlp_params - lm_head_params - ln_params

print("Parameter Breakdown by Component:")
print(f"  Embeddings (wte + wpe):  {emb_params:>10,}  ({emb_params/total_params*100:5.1f}%)")
print(f"  Attention (QKV + proj):  {attn_params:>10,}  ({attn_params/total_params*100:5.1f}%)")
print(f"  MLP (fc + proj):         {mlp_params:>10,}  ({mlp_params/total_params*100:5.1f}%)")
print(f"  LM Head:                 {lm_head_params:>10,}  ({lm_head_params/total_params*100:5.1f}%)")
print(f"  Layer Norms:             {ln_params:>10,}  ({ln_params/total_params*100:5.1f}%)")
print(f"  Other (dropout etc):     {other_params:>10,}  ({other_params/total_params*100:5.1f}%)")
print(f"  {'─' * 30}")
print(f"  TOTAL:                   {total_params:>10,}")

# -------------------------------------------------------------------
# Per-block breakdown (how params are distributed across transformer layers)
# -------------------------------------------------------------------
print()
print("Per-Block Parameter Distribution:")
num_blocks = config.n_layer
for i in range(num_blocks):
    block_params = 0
    for name, p in model.named_parameters():
        if f'transformer.h.{i}.' in name:
            block_params += p.numel()
    print(f"  Block {i}:  {block_params:>10,} params  "
          f"({block_params/total_params*100:5.1f}% of total)")

GPT-2-like Model Architecture — Layer-by-Layer Breakdown

├─ transformer               (GPT2Model         )  params: 44,703,744 / 44,703,744  (170.53 MB)
  │  ├─ wte                       (Embedding         )  params: 25,731,584 / 25,731,584  ( 98.16 MB)  ← weight: [50257, 512]
  │  ├─ wpe                       (Embedding         )  params:     56,832 /     56,832  (  0.22 MB)  ← weight: [111, 512]
  │  ├─ drop                      (Dropout           )  params:          0 /          0  (  0.00 MB)
  │  ├─ h                         (ModuleList        )  params: 18,914,304 / 18,914,304  ( 72.15 MB)
    │  ├─ 0                         (GPT2Block         )  params:  3,152,384 /  3,152,384  ( 12.03 MB)
      │  ├─ ln_1                      (LayerNorm         )  params:      1,024 /      1,024  (  0.00 MB)  ← weight: [512]
      │  ├─ attn                      (GPT2Attention     )  params:  1,050,624 /  1,050,624  (  4.01 MB)
        │  ├─ c_attn                    (Conv1D            )  para

## 6. Resize Token Embeddings

Although GPT-2's default vocab size is 50,257 (same as our config), calling `resize_token_embeddings` ensures the embedding matrix and LM head are correctly sized for our tokenizer. This is a safety step — if the tokenizer had a different vocab size (e.g., after adding special tokens), this would handle it automatically.

In [9]:
# Ensure the model's embedding matrix matches the tokenizer's vocabulary size
model.resize_token_embeddings(len(tokenizer))
print(f"Tokenizer vocab size: {len(tokenizer)}")

Tokenizer vocab size: 50257


## 7. Check Hardware & Move Model to GPU

Verify CUDA is available and move the model to the GPU. On an RTX 5090 with 24 GB VRAM, a 110M-parameter model fits comfortably — we can use `fp16` mixed precision for faster training.

In [10]:
import torch

# -------------------------------------------------------------------
# 7. Check hardware and move model to GPU
# -------------------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

# Move model to GPU (or fall back to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Model device:", next(model.parameters()).device)

CUDA available: True
Device count: 1
Device name: NVIDIA GeForce RTX 5090 Laptop GPU
VRAM (GB): 25.650855936
Model device: cuda:0


## 8. Configure Data Collator & Training Arguments

### Data Collator
`DataCollatorForLanguageModeling` with `mlm=False` sets up **causal language modeling** (CLM): the model predicts the next token given all previous tokens. Labels are automatically created by shifting the input IDs by one position.

### Training Arguments
We use the validation split to monitor overfitting — the trainer evaluates on the val set every 100 steps.

| Argument | Value | Purpose |
|----------|-------|---------|
| `num_train_epochs` | 3 | Full passes over the train set |
| `per_device_train_batch_size` | 8 | Batch size per GPU (adjust based on VRAM) |
| `eval_strategy` | "steps" | Evaluate on val set periodically |
| `eval_steps` | 100 | Run validation every 100 steps |
| `warmup_steps` | 100 | Linear LR warmup to avoid early instability |
| `weight_decay` | 0.01 | L2 regularization (AdamW) |
| `fp16` | True | Mixed precision — halves memory, ~2× speedup on RTX 5090 |
| `logging_steps` | 25 | Log loss more frequently (cleaner output) |
| `save_total_limit` | 2 | Keep only the 2 most recent checkpoints |
| `load_best_model_at_end` | True | Restore best checkpoint after training |
| `report_to` | "none" | Disable WandB/TensorBoard for simplicity |

In [11]:
# -------------------------------------------------------------------
# 8. Prepare data collator and training arguments
# -------------------------------------------------------------------

# Data collator: automatically creates labels for causal LM
# mlm=False → causal LM (predict next token), not masked LM (predict masked tokens)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Training arguments — with validation monitoring
training_args = TrainingArguments(
    output_dir="./models/gpt2-small-twitter-checkpoint",  # Checkpoints saved here
    num_train_epochs=3,             # Small dataset → few epochs to avoid overfitting
    per_device_train_batch_size=8,  # Batch size; RTX 5090 24GB can handle 8 with fp16
    per_device_eval_batch_size=8,   # Evaluation batch size
    eval_strategy="steps",          # Evaluate on val set periodically
    eval_steps=100,                 # Every 100 steps (≈ every 2% of an epoch)
    gradient_accumulation_steps=1,
    warmup_steps=100,               # Gradually increase LR for first 100 steps
    weight_decay=0.01,              # AdamW weight decay for regularization
    logging_steps=25,               # Log loss every 25 steps
    save_steps=500,                 # Save checkpoint every 500 steps
    save_total_limit=2,             # Keep at most 2 checkpoints (saves disk space)
    prediction_loss_only=True,      # Only compute loss (don't generate during eval)
    load_best_model_at_end=True,    # Restore best checkpoint after training
    metric_for_best_model="eval_loss",  # Use val loss to determine "best"
    fp16=True,                      # Mixed precision: faster training, less memory
    dataloader_num_workers=4,       # Parallel data loading
    report_to="none",               # Disable external logging (no WandB/TensorBoard)
)

## 9. Create Trainer & Start Training

The Hugging Face `Trainer` handles the entire training loop: forward pass, loss computation, backpropagation, optimizer step, learning rate scheduling, checkpointing, and logging. We pass `eval_dataset` so the trainer can track validation loss and restore the best model.

> ⏱️ ~2 minutes on RTX 5090 with 5K train + 1K val examples, ~70M parameters, fp16.

In [12]:
# -------------------------------------------------------------------
# 9. Create Trainer and start training (with validation)
# -------------------------------------------------------------------
trainer = Trainer(
    model=model,                        # Our GPT2LMHeadModel (random init)
    args=training_args,                 # TrainingArguments defined above
    train_dataset=tokenized_train,      # 5000 training examples
    eval_dataset=tokenized_val,         # 1000 validation examples
    data_collator=data_collator,        # Causal LM data collator
    processing_class=tokenizer,         # Tokenizer (renamed from 'tokenizer' in transformers 5.x)
)

# Start training — this runs the full training loop with periodic eval
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,5.171437,4.317164
200,3.180141,3.078187
300,2.897153,2.950860
400,2.772235,2.885640
500,2.812278,2.831942
600,2.802492,2.787280
700,2.776563,2.761002
800,2.707159,2.736545
900,2.661868,2.723326
1000,2.602636,2.708815


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.20it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1875, training_loss=2.9916643778483074, metrics={'train_runtime': 406.7496, 'train_samples_per_second': 36.878, 'train_steps_per_second': 4.61, 'total_flos': 188964126720000.0, 'train_loss': 2.9916643778483074, 'epoch': 3.0})

## 10. Save the Trained Model

Save both the model weights and tokenizer to disk. The output directory (`./models/gpt2-small-twitter`) will contain:
- `config.json` — model architecture configuration
- `model.safetensors` — model weights in safetensors format
- `tokenizer.json`, `vocab.json`, `merges.txt` — tokenizer files

In [13]:
# -------------------------------------------------------------------
# 10. Save the trained model and tokenizer
# -------------------------------------------------------------------
# Output directory: tutorials/01-llm-transformer-training/models/gpt2-small-twitter/
model.save_pretrained("./models/gpt2-small-twitter")
tokenizer.save_pretrained("./models/gpt2-small-twitter")
print("Model and tokenizer saved to ./models/gpt2-small-twitter")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.08it/s]

Model and tokenizer saved to ./models/gpt2-small-twitter


## 11. Evaluate the Trained Model on the Test Set

Now we evaluate on the **held-out test set** (1,000 examples the model never saw during training). For 5 randomly selected examples we show:

1. **Loss & perplexity** on the full formatted text — lower perplexity = model is more confident
2. **Generated** sentiment via greedy decoding (only the prompt up to `"Output:"` is fed)
3. **Extract** the first word after `"Output:"` as the predicted sentiment

After the examples, we compute **aggregate benchmarks** across the entire test set.

> **💡 Perplexity** = $e^{\text{loss}}$. Measures prediction confidence — perplexity of 1 means perfect prediction; higher values = more uncertain.

In [14]:
import pandas as pd
import torch
import random
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# -------------------------------------------------------------------
# 1. Load the saved model and tokenizer
# -------------------------------------------------------------------
model_path = "./models/gpt2-small-twitter"
model = GPT2LMHeadModel.from_pretrained(model_path)
tokenizer = GPT2Tokenizer.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set to evaluation mode (disables dropout)
print(f"Model loaded on {device}")

# -------------------------------------------------------------------
# 2. Rebuild test texts (must match training preprocessing)
# -------------------------------------------------------------------
df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv')
df = df.head(7000).copy()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_test = df.iloc[6000:7000]  # Same 1000 test examples as during tokenization

instruction = "Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:"
fixed_template = f"Instruction: {instruction}\nInput:\nOutput:"
fixed_token_count = len(tokenizer.encode(fixed_template))
max_total_tokens = fixed_token_count + 80  # = 111 (matches training n_positions)
max_tweet_tokens = max_total_tokens - fixed_token_count - 1

def truncate_tweet(tweet, max_tokens):
    tokens = tokenizer.encode(tweet, truncation=True, max_length=max_tokens)
    return tokenizer.decode(tokens, skip_special_tokens=True)

test_texts = []
for _, row in df_test.iterrows():
    st = truncate_tweet(row['content'], max_tweet_tokens)
    test_texts.append(f"Instruction: {instruction}\nInput: {st}\nOutput: {row['sentiment']}")

print(f"Loaded {len(test_texts)} test examples")

# -------------------------------------------------------------------
# 3. Show 5 random examples with predictions
# -------------------------------------------------------------------
random.seed(42)
indices = random.sample(range(len(test_texts)), 5)

for idx in indices:
    full_text = test_texts[idx]
    tweet = df_test.iloc[idx]['content']
    expected = df_test.iloc[idx]['sentiment']
    short_tweet = truncate_tweet(tweet, max_tweet_tokens)

    print(f"\n--- Test Example {idx} ---")
    print(f"  Tweet: {tweet[:90]}...")
    print(f"  Expected: {expected}")

    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_total_tokens)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        ppl = torch.exp(outputs.loss)
    print(f"  Loss: {outputs.loss.item():.4f} | Perplexity: {ppl.item():.1f}")

    prompt = f"Instruction: {instruction}\nInput: {short_tweet}\nOutput:"
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True)
    prompt_ids = {k: v.to(device) for k, v in prompt_ids.items()}
    available = model.config.n_positions - prompt_ids["input_ids"].shape[1]

    if available > 0:
        with torch.no_grad():
            gen = model.generate(prompt_ids["input_ids"], max_new_tokens=min(5, available),
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
                                 eos_token_id=tokenizer.eos_token_id)
        gen_text = tokenizer.decode(gen[0], skip_special_tokens=True)
        if "Output:" in gen_text:
            pred = gen_text.split("Output:")[-1].strip().split()[0] if gen_text.split("Output:")[-1].strip() else "<empty>"
        else:
            pred = gen_text.strip().split()[0] if gen_text.strip() else "<empty>"
        match = "✓" if pred == expected else "✗"
        print(f"  Predicted: {pred} {match}")

# -------------------------------------------------------------------
# 4. Aggregate benchmark across the test set
# -------------------------------------------------------------------
print("\n" + "=" * 60)
print("📊 Test Set Benchmark (1000 examples)")
print("=" * 60)

total_loss = 0
total_ppl = 0
correct = 0
label_map = {"negative": 0, "neutral": 1, "positive": 2, "sadness": 0, "worry": 0, "surprise": 2,
             "love": 2, "empty": 1, "hate": 0, "anger": 0, "relief": 2, "enthusiasm": 2, "fun": 2}
valid_labels = {"negative", "neutral", "positive"}

for i, full_text in enumerate(test_texts):
    expected = df_test.iloc[i]['sentiment']

    # Loss
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_total_tokens)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        total_loss += outputs.loss.item()
        total_ppl += torch.exp(outputs.loss).item()

    # Quick generation (first 200 only to save time, sample 100)
    if i < 200:
        prompt = f"Instruction: {instruction}\nInput: {truncate_tweet(df_test.iloc[i]['content'], max_tweet_tokens)}\nOutput:"
        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_total_tokens-1)
        prompt_ids = {k: v.to(device) for k, v in prompt_ids.items()}
        available = model.config.n_positions - prompt_ids["input_ids"].shape[1]
        if available > 0:
            with torch.no_grad():
                gen = model.generate(prompt_ids["input_ids"], max_new_tokens=min(5, available),
                                     do_sample=False, pad_token_id=tokenizer.eos_token_id,
                                     eos_token_id=tokenizer.eos_token_id)
            gen_text = tokenizer.decode(gen[0], skip_special_tokens=True)
            if "Output:" in gen_text:
                pred = gen_text.split("Output:")[-1].strip().split()[0] if gen_text.split("Output:")[-1].strip() else ""
            else:
                pred = ""
            # Map to coarse sentiment
            mapped = label_map.get(pred, -1)
            exp_mapped = label_map.get(expected, -1)
            if mapped == exp_mapped and mapped >= 0:
                correct += 1

n = len(test_texts)
acc = correct / min(200, n) * 100
print(f"  Avg Test Loss:       {total_loss/n:.4f}")
print(f"  Avg Test Perplexity:  {total_ppl/n:.1f}")
print(f"  Accuracy (200-sample): {acc:.1f}%  ({correct}/{min(200,n)} coarse-grained matches)")
print(f"  Note: Accuracy is coarse-grained (negative/neutral/positive); full test would differ.")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 8501.36it/s]


Model loaded on cuda
Loaded 1000 test examples

--- Test Example 654 ---
  Tweet: Ordered a new computer! Unfortunately it won't get here until the end of June....
  Expected: sadness
  Loss: 2.3109 | Perplexity: 10.1
  Predicted: <empty> ✗

--- Test Example 114 ---
  Tweet: ah thats better snow patrol! now to be stuck indoors  still weekend coming up and fresh ai...
  Expected: happiness
  Loss: 3.5090 | Perplexity: 33.4
  Predicted: worry ✗

--- Test Example 25 ---
  Tweet: is at home with a pukey boy! Poor little baby...
  Expected: worry
  Loss: 2.0433 | Perplexity: 7.7
  Predicted: worry ✓

--- Test Example 759 ---
  Tweet: urgh, i really hate that medicine...
  Expected: hate
  Loss: 1.4245 | Perplexity: 4.2
  Predicted: worry ✗

--- Test Example 281 ---
  Tweet: @nickkk_ that sucks!...
  Expected: sadness
  Loss: 1.1759 | Perplexity: 3.2
  Predicted: neutral ✗

📊 Test Set Benchmark (1000 examples)
  Avg Test Loss:       2.5623
  Avg Test Perplexity:  18.9
  Accuracy (200-sample)

---

### ✅ Part 1 Summary

We've successfully trained a GPT-2-style causal language model **from randomly initialized weights** using Hugging Face's high-level APIs:
- Configured a ~70M-parameter GPT-2 model with `GPT2Config` (random weights, not pretrained)
- Tokenized the Twitter sentiment dataset with GPT-2's pretrained BPE tokenizer
- Used **proper train (5000) / validation (1000) / test (1000)** splits
- Trained for 3 epochs with mixed precision (fp16) + validation monitoring on an RTX 5090
- Saved the best model (by validation loss) and evaluated on the held-out test set

> **⚠️ Reminder:** The GPT-2 weights are randomly initialized (native training), not fine-tuned. Only the tokenizer is borrowed from pretrained GPT-2.

---

---
## 🔴 Part 2 — Building GPT-2 from Scratch in PyTorch

In Part 2 we **reimplement the entire GPT-2 architecture** from the ground up using pure PyTorch. This is the educational core of the tutorial — by building each component yourself, you'll deeply understand:

- How **multi-head causal self-attention** works (the heart of transformers)
- How the **feed-forward MLP** expands and projects hidden states
- How **layer normalization + residual connections** stabilize training
- How **learned positional embeddings** encode sequence order
- How the **LM head** maps hidden states to vocabulary probabilities

We then train this custom model with a **manual training loop** (no Trainer API) and a **custom greedy generation function**.

### Architecture Overview

```
Input IDs (B, T)
     │
     ▼
┌─────────────────────┐
│  Token Embedding    │  wte: (vocab_size, n_embd)
│  + Position Embed   │  wpe: (n_positions, n_embd)
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  GPT2Block × N      │  ← repeated n_layer times
│  ┌────────────────┐ │
│  │ LayerNorm       │ │
│  │ Multi-Head      │ │  Q, K, V projections → scaled dot-product attention
│  │ Causal Attn     │ │  + causal mask (upper triangular)
│  │   + Residual    │ │
│  ├────────────────┤ │
│  │ LayerNorm       │ │
│  │ MLP (4× expand) │ │  Linear → GELU → Linear
│  │   + Residual    │ │
│  └────────────────┘ │
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  Final LayerNorm    │  ln_f
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│  LM Head (Linear)   │  lm_head: (n_embd, vocab_size)
│  → Logits (B,T,V)   │  (weight-tied with wte)
└─────────────────────┘
```

---

> **💡 Note:** We build a **smaller** model in Part 2 (~18M params, `n_embd=256, n_layer=6, n_head=8`) for faster experimentation. The architecture is identical to Part 1's model — only the hyperparameters differ. See the label-distribution diagnosis cell above for why 256-dim is needed to avoid mode collapse.

## 11. Clean Slate — Remove HF's GPT2LMHeadModel

Before defining our own `GPT2LMHeadModel`, we must unload the Hugging Face version to avoid naming conflicts. We also import the core PyTorch modules we'll need.

In [15]:
# Remove Hugging Face's GPT2LMHeadModel so we can define our own
del GPT2LMHeadModel

# Core PyTorch imports for building the model from scratch
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

## 12. Multi-Head Causal Self-Attention

This is the **most important component** of the transformer. Here's how it works:

### Step-by-step:

1. **Project** input `x` into **Q** (query), **K** (key), and **V** (value) with a single linear layer `c_attn`
2. **Split** into `n_head` independent heads, each with dimension `head_dim = n_embd / n_head`
3. **Compute attention scores**: $\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right)V$
4. **Apply causal mask** — an upper-triangular matrix of $-\infty$ so position $i$ can only attend to positions $\leq i$ (prevents "cheating" by looking at future tokens)
5. **Apply optional padding mask** — ignore padding tokens in the attention calculation
6. **Concatenate** heads back and project with `c_proj`

### Key dimensions:
- Input: `(B, T, C)` → Batch × Tokens × Embedding
- Q/K/V: `(B, n_head, T, head_dim)`
- Attention weights: `(B, n_head, T, T)`
- Output: `(B, T, C)`

In [16]:
class GPT2Attention(nn.Module):
    """
    Multi-Head Causal Self-Attention layer (GPT-2 style).

    Projects input into Q, K, V; splits into multiple heads;
    computes scaled dot-product attention with a causal mask and optional padding mask.
    """
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = self.n_embd // self.n_head
        assert self.head_dim * self.n_head == self.n_embd, \
            "n_embd must be divisible by n_head"

        # Combined Q, K, V projection: (n_embd) → (3 * n_embd)
        self.c_attn = nn.Linear(self.n_embd, 3 * self.n_embd, bias=True)
        # Output projection: (n_embd) → (n_embd)
        self.c_proj = nn.Linear(self.n_embd, self.n_embd, bias=True)

        # Dropout regularizers
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

        # Causal mask: lower-triangular matrix of 1s
        # Shape: (1, 1, n_positions, n_positions)
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.n_positions, config.n_positions))
                 .view(1, 1, config.n_positions, config.n_positions)
        )

    def forward(self, x, attention_mask=None):
        """
        Args:
            x:              (B, T, C) input hidden states
            attention_mask: (B, T) padding mask (1=real, 0=pad), or None
        Returns:
            y: (B, T, C) attention output
        """
        B, T, C = x.size()

        # ---- 1. Compute Q, K, V ----
        qkv = self.c_attn(x)                      # (B, T, 3*C)
        q, k, v = qkv.split(self.n_embd, dim=2)   # Each: (B, T, C)

        # ---- 2. Reshape to multi-head: (B, n_head, T, head_dim) ----
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # ---- 3. Scaled dot-product attention ----
        att = (q @ k.transpose(-2, -1)) * (1.0 / (self.head_dim ** 0.5))  # (B, n_head, T, T)

        # ---- 4. Apply causal mask ----
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))

        # ---- 5. Apply padding mask (if provided) ----
        # Standard GPT-2 convention: mask both query AND key dimensions
        # so padding tokens neither attend nor are attended to
        if attention_mask is not None:
            # attention_mask: (B, T) → (B, 1, 1, T) for key masking
            key_mask = attention_mask[:, None, None, :]        # (B, 1, 1, T)
            att = att.masked_fill(key_mask == 0, float('-inf'))

        # ---- 6. Softmax + dropout + weighted sum ----
        att = F.softmax(att, dim=-1)            # (B, n_head, T, T)
        att = self.attn_dropout(att)
        y = att @ v                              # (B, n_head, T, head_dim)

        # ---- 7. Concatenate heads and project output ----
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

## 13. Feed-Forward Network (MLP)

The MLP block follows the standard transformer pattern:
- **Expand** the hidden dimension by 4× (`n_embd` → `4 * n_embd`)
- Apply **GELU** activation (a smooth ReLU variant used in GPT-2)
- **Project** back to `n_embd`

This two-layer MLP is applied **independently** to each token position (no cross-token interaction — that's the attention layer's job).

In [17]:
class GPT2MLP(nn.Module):
    """
    GPT-2 Feed-Forward Network: Linear → GELU → Linear → Dropout.

    Expands hidden dim by 4×, applies GELU activation, then projects back.
    Applied independently to each token position.
    """
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=True)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=True)
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.act(self.c_fc(x))        # (B, T, 4C)
        x = self.dropout(self.c_proj(x))   # (B, T, C)
        return x

## 14. GPT-2 Transformer Block

Each transformer block combines **attention** and **MLP** with **pre-norm residual connections**:

$$\text{out} = x + \text{Attn}(\text{LayerNorm}(x))$$
$$\text{out} = \text{out} + \text{MLP}(\text{LayerNorm}(\text{out}))$$

> **Why pre-norm?** GPT-2 applies LayerNorm **before** each sublayer (not after, as in the original transformer). This is more stable during training and is standard in modern LLMs.

In [18]:
class GPT2Block(nn.Module):
    """
    A single GPT-2 transformer (decoder) block.

    Architecture (pre-norm residual):
        x = x + Attention(LayerNorm(x))
        x = x + MLP(LayerNorm(x))
    """
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)   # Norm before attention
        self.attn = GPT2Attention(config)          # Multi-head causal self-attention
        self.ln_2 = nn.LayerNorm(config.n_embd)   # Norm before MLP
        self.mlp = GPT2MLP(config)                 # Feed-forward network

    def forward(self, x, attention_mask=None):
        # Pre-norm + attention + residual
        x = x + self.attn(self.ln_1(x), attention_mask)
        # Pre-norm + MLP + residual
        x = x + self.mlp(self.ln_2(x))
        return x

## 15. GPT-2 Base Model (Embeddings + Transformer Blocks)

This is the **core transformer** — it maps token IDs to hidden states through:

1. **Token embeddings** (`wte`): Each token ID → a learned vector of size `n_embd`
2. **Position embeddings** (`wpe`): Each position (0, 1, 2, ...) → a learned vector of size `n_embd`
3. **Sum**: `x = token_emb + pos_emb` (broadcast position embeddings across the batch)
4. **Transformer blocks**: Stack of `n_layer` `GPT2Block`s
5. **Final LayerNorm**: One last normalization before the LM head

> **📐 Shape tracking:** `(B, T)` token IDs → `(B, T, n_embd)` after embeddings → `(B, T, n_embd)` after transformer → `(B, T, vocab_size)` after LM head.

In [19]:
class GPT2Model(nn.Module):
    """
    GPT-2 base transformer model: embeddings → dropout → blocks → final layer norm.

    Does NOT include the LM head — that's added by GPT2LMHeadModel.
    """
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embedding: maps token IDs to vectors
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        # Position embedding: maps position indices to vectors
        self.wpe = nn.Embedding(config.n_positions, config.n_embd)

        # Embedding dropout (standard in GPT-2 — applied after wte + wpe)
        self.drop = nn.Dropout(config.embd_pdrop)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([GPT2Block(config) for _ in range(config.n_layer)])

        # Final layer normalization (before LM head)
        self.ln_f = nn.LayerNorm(config.n_embd, eps=1e-5)

    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids:      (B, T) token IDs
            attention_mask: (B, T) optional padding mask (1=real, 0=pad)
        Returns:
            x: (B, T, n_embd) hidden states
        """
        B, T = input_ids.size()
        assert T <= self.config.n_positions, \
            f"Sequence length {T} exceeds n_positions {self.config.n_positions}"

        # ---- Token embeddings: (B, T) → (B, T, n_embd) ----
        token_embeds = self.wte(input_ids)

        # ---- Position embeddings: create position IDs [0, 1, ..., T-1] ----
        position_ids = torch.arange(T, device=input_ids.device).unsqueeze(0)  # (1, T)
        pos_embeds = self.wpe(position_ids)  # (1, T, n_embd)

        # ---- Sum + dropout ----
        x = self.drop(token_embeds + pos_embeds)  # (B, T, n_embd)

        # ---- Pass through each transformer block ----
        for block in self.blocks:
            x = block(x, attention_mask)

        # ---- Final layer norm ----
        x = self.ln_f(x)  # (B, T, n_embd)
        return x

## 16. GPT-2 LM Head Model (Complete Trainable Model)

The `GPT2LMHeadModel` wraps the base transformer with a **language modeling head**:

- **`lm_head`**: A linear layer `(n_embd → vocab_size)` that maps hidden states to logits over the vocabulary
- **Weight tying**: `lm_head.weight` is tied to `wte.weight` — this saves parameters and improves training (same matrix is used for input embedding and output projection)
- **Loss**: Standard cross-entropy loss with **label shifting** — the model predicts token $n+1$ given token $n$, so we shift logits by 1 position

### Label Shifting (Causal LM)

```
Input:   [tok0, tok1, tok2, ..., tok_{T-1}]
Logits:  [p(tok1|tok0), p(tok2|tok0,tok1), ..., p(tok_T|...)]
Labels:  [tok1, tok2, tok3, ..., tok_T]
```

We compute `cross_entropy(logits[:-1], labels[1:])` — predicting each next token.

In [20]:
class GPT2LMHeadModel(nn.Module):
    """
    Complete GPT-2 model with language modeling head.

    Architecture: GPT2Model → lm_head (Linear) → logits
    Computes causal LM loss with label shifting.
    Returns a dict for compatibility.
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = GPT2Model(config)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying: share weights between input embedding and output projection
        self.lm_head.weight = self.transformer.wte.weight

        # GPT-2 standard initialization: Normal(0, 0.02), bias→0, LayerNorm→(1, 0)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """GPT-2 style weight initialization."""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if hasattr(module, 'bias') and module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        Args:
            input_ids:      (B, T) token IDs
            attention_mask: (B, T) optional padding mask (1=real, 0=pad)
            labels:         (B, T) optional target token IDs for loss
        Returns:
            dict with "loss" (if labels provided) and "logits"
        """
        hidden_states = self.transformer(input_ids, attention_mask)  # (B, T, n_embd)
        logits = self.lm_head(hidden_states)                         # (B, T, vocab_size)

        loss = None
        if labels is not None:
            # Causal LM: shift logits and labels by 1 position
            shift_logits = logits[..., :-1, :].contiguous()     # (B, T-1, V)
            shift_labels = labels[..., 1:].contiguous()          # (B, T-1)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

## 17. Configure the Custom Model & Training Setup

We define a **smaller** model for Part 2 (~25M parameters) to make manual training faster:

| Parameter | Part 1 (HF) | Part 2 (Custom) |
|-----------|-------------|-----------------|
| `n_embd` | 512 | 256 |
| `n_layer` | 6 | 6 |
| `n_head` | 8 | 8 |
| Approx. params | ~70M | ~25M |

> **⚠️ Why 25M and not 7M?** With 13 fine-grained sentiment labels on a small dataset, a 7M-parameter model collapses to the majority class ("worry" = 33%). Doubling the embedding dimension and layer count gives enough capacity to distinguish labels. See the diagnosis cell above for the label distribution.

In [21]:
# ============================================================================
# 17. Part 2: Data Pipeline + Model Config (self-contained)
# ============================================================================

# --- 17a. Rebuild tokenized splits (same seed as Part 1) ---
import pandas as pd
df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv').head(7000)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_train = df.iloc[:5000]; df_val = df.iloc[5000:6000]; df_test = df.iloc[6000:7000]

instruction = "Analyze the sentiment of the following tweet. Output exactly one word, with no punctuation or extra text:"
fixed_token_count = len(tokenizer.encode(f"Instruction: {instruction}\nInput:\nOutput:"))
max_total_tokens = fixed_token_count + 80   # = 111
max_tweet_tokens = max_total_tokens - fixed_token_count - 1

def truncate_tweet(tweet, max_tokens):
    tokens = tokenizer.encode(tweet, truncation=True, max_length=max_tokens)
    return tokenizer.decode(tokens, skip_special_tokens=True)

def tokenize_split(df_split, name):
    texts = [f"Instruction: {instruction}\nInput: {truncate_tweet(r['content'], max_tweet_tokens)}\nOutput: {r['sentiment']}"
             for _, r in df_split.iterrows()]
    ds = Dataset.from_dict({"text": texts})
    ds = ds.map(lambda ex: tokenizer(ex["text"], truncation=True, padding="max_length",
                                      max_length=max_total_tokens),
                batched=True, remove_columns=["text"])
    print(f"  {name}: {len(ds)} examples, avg real tokens: {sum(sum(s['attention_mask']) for s in ds)/len(ds):.1f}")
    return ds

print("Part 2 data pipeline:")
tok_train = tokenize_split(df_train, "Train")
tok_val   = tokenize_split(df_val,   "Val  ")
tok_test  = tokenize_split(df_test,  "Test ")

# --- 17b. Inspect one train example ---
print(f"\n--- Train sample 0 ---")
s = tok_train[0]
real = sum(s["attention_mask"])
print(f"  Total len: {len(s['input_ids'])} | Real: {real} | Pad: {len(s['input_ids'])-real}")
print(f"  First 12 IDs: {s['input_ids'][:12]}")
print(f"  Last  10 IDs: {s['input_ids'][-10:]}")
print(f"  Attn mask (first 12, last 10): {s['attention_mask'][:12]} ... {s['attention_mask'][-10:]}")
print(f"  Decoded: {tokenizer.decode(s['input_ids'], skip_special_tokens=True)[:100]}...")

# --- 17c. Model config ---
class Config:
    vocab_size  = 50257
    n_positions = max_total_tokens   # = 111
    n_embd      = 256                # Double from 128 — needs enough capacity for 13 labels
    n_layer     = 6                  # Match Part 1 layer count
    n_head      = 8                  # head_dim = 256/8 = 32
    attn_pdrop  = 0.1
    resid_pdrop = 0.1
    embd_pdrop  = 0.1

config = Config()
model = GPT2LMHeadModel(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {total_params:,} params on {device}")

# --- 17d. Dataloader + collator ---
from torch.utils.data import DataLoader
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Inspect one batch
temp_loader = DataLoader(tok_train, batch_size=4, shuffle=False, collate_fn=data_collator)
batch = next(iter(temp_loader))
print(f"\nBatch inspection:")
print(f"  input_ids: {list(batch['input_ids'].shape)}")
print(f"  labels:    {list(batch['labels'].shape)}")
print(f"  attn_mask: {list(batch['attention_mask'].shape)}")
print(f"  Real tokens per sample: {batch['attention_mask'].sum(dim=1).tolist()}")
print(f"  input_ids[0,:15]  = {batch['input_ids'][0,:15].tolist()}")
print(f"  labels[0,:15]     = {batch['labels'][0,:15].tolist()}  (should match)")
print(f"  Decoded sample 0: {tokenizer.decode(batch['input_ids'][0], skip_special_tokens=True)[:80]}...")

Part 2 data pipeline:


Map: 100%|██████████| 5000/5000 [00:00<00:00, 31809.97 examples/s]


  Train: 5000 examples, avg real tokens: 52.4


Map: 100%|██████████| 1000/1000 [00:00<00:00, 22252.48 examples/s]


  Val  : 1000 examples, avg real tokens: 52.4


Map: 100%|██████████| 1000/1000 [00:00<00:00, 22620.44 examples/s]


  Test : 1000 examples, avg real tokens: 52.2

--- Train sample 0 ---
  Total len: 111 | Real: 51 | Pad: 60
  First 12 IDs: [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13]
  Last  10 IDs: [50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]
  Attn mask (first 12, last 10): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ... [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Decoded: Instruction: Analyze the sentiment of the following tweet. Output exactly one word, with no punctuat...

Model: 17,633,280 params on cuda

Batch inspection:
  input_ids: [4, 111]
  labels:    [4, 111]
  attn_mask: [4, 111]
  Real tokens per sample: [51, 44, 70, 52]
  input_ids[0,:15]  = [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13, 25235, 3446, 530]
  labels[0,:15]     = [6310, 2762, 25, 16213, 2736, 262, 15598, 286, 262, 1708, 6126, 13, 25235, 3446, 530]  (should match)
  Decoded sample 0: Instruction: Analyze the sentiment of the following tweet. Output exactly one wo...


## 18. Manual Training Loop (with Validation)

Unlike Part 1 where we used the `Trainer` API, here we write the training loop **by hand** to understand each step. We also compute **validation loss** at the end of each epoch to monitor overfitting.

**Per-step:**
1. **Forward pass**: `model(input_ids, labels=input_ids)` → computes logits + loss
2. **Backward pass**: `loss.backward()` → computes gradients
3. **Optimizer step**: `optimizer.step()` → updates weights
4. **Zero gradients**: `optimizer.zero_grad()` → reset for next batch

**Per-epoch:** Evaluate on the validation split (no gradients, `model.eval()`).

> **💡 Key insight:** For causal LM, the labels are the **same as the input IDs** — the model learns to predict each token from all previous tokens in the sequence.

In [22]:
# ============================================================================
# 18. Manual Training Loop with Validation
# ============================================================================

train_loader = DataLoader(tok_train, batch_size=32, shuffle=True,  collate_fn=data_collator)
val_loader   = DataLoader(tok_val,   batch_size=32, shuffle=False, collate_fn=data_collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)  # moderate LR for 25M model
num_epochs = 15

print(f"{'Epoch':<8} {'Train Loss':>12} {'Val Loss':>12} {'Val PPL':>10}")
print("-" * 45)

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()
    train_loss = 0; n_train = 0
    for batch in train_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)           # DataCollator provides labels

        out = model(ids, attention_mask=mask, labels=lbls)
        loss = out["loss"]
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_loss += loss.item(); n_train += 1

    # ---- Validate ----
    model.eval()
    val_loss = 0; n_val = 0
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)
            out = model(ids, attention_mask=mask, labels=lbls)
            val_loss += out["loss"].item(); n_val += 1

    avg_train = train_loss / n_train
    avg_val   = val_loss / n_val
    val_ppl   = torch.exp(torch.tensor(avg_val)).item()
    print(f"{epoch+1:<8} {avg_train:>12.4f} {avg_val:>12.4f} {val_ppl:>10.1f}")

print("\nTraining complete!")

Epoch      Train Loss     Val Loss    Val PPL
---------------------------------------------
1              4.4889       2.9426       19.0
2              2.7529       2.7212       15.2
3              2.5449       2.6485       14.1
4              2.4100       2.6150       13.7
5              2.3114       2.6147       13.7
6              2.2174       2.6171       13.7
7              2.1296       2.6358       14.0
8              2.0439       2.6580       14.3
9              1.9608       2.6861       14.7
10             1.8798       2.7159       15.1
11             1.7992       2.7518       15.7
12             1.7215       2.8078       16.6
13             1.6479       2.8386       17.1
14             1.5758       2.8757       17.7
15             1.5030       2.9291       18.7

Training complete!


## 19. Evaluate the Custom Model on the Test Set

Same evaluation protocol as Part 1, but using our custom greedy `generate()` function (since this is NOT a Hugging Face model — no `.generate()` method). We evaluate on the **held-out test split**.

In [23]:
# ============================================================================
# 19. Greedy generation + test-set evaluation
# ============================================================================

def generate(model, prompt_ids, max_new_tokens, eos_token_id):
    """
    Greedy autoregressive decoding with proper attention mask.
    Each new token has attention_mask=1 (real token, not padding).
    """
    generated = prompt_ids.clone()                    # (1, cur_len)
    for _ in range(max_new_tokens):
        cur_len = generated.size(1)
        # attention_mask: all ones (no padding during generation)
        attn_mask = torch.ones(1, cur_len, device=generated.device)
        with torch.no_grad():
            out = model(generated, attention_mask=attn_mask)
            logits = out["logits"]                    # (1, cur_len, V)
            next_tok = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated = torch.cat([generated, next_tok], dim=-1)
        if (next_tok == eos_token_id).any():
            break
    return generated

# --- Evaluate on test set ---
print("=" * 60)
print("Part 2 Test Set Benchmark (1000 examples)")
print("=" * 60)

model.eval()
total_loss = 0
for i in range(len(df_test)):
    st = truncate_tweet(df_test.iloc[i]['content'], max_tweet_tokens)
    full_text = f"Instruction: {instruction}\nInput: {st}\nOutput: {df_test.iloc[i]['sentiment']}"
    enc = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_total_tokens)
    ids  = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        out = model(ids, attention_mask=mask, labels=ids)
        total_loss += out["loss"].item()

avg_loss = total_loss / len(df_test)
avg_ppl  = torch.exp(torch.tensor(avg_loss)).item()
print(f"  Avg Test Loss:       {avg_loss:.4f}")
print(f"  Avg Test Perplexity:  {avg_ppl:.1f}")

# --- Show 5 examples ---
print(f"\n--- 5 Test Examples ---")
import random; random.seed(42)
for idx in random.sample(range(len(df_test)), 5):
    tweet = df_test.iloc[idx]['content']
    expected = df_test.iloc[idx]['sentiment']
    st = truncate_tweet(tweet, max_tweet_tokens)

    # Loss
    full_text = f"Instruction: {instruction}\nInput: {st}\nOutput: {expected}"
    enc = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_total_tokens)
    ids  = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        out = model(ids, attention_mask=mask, labels=ids)
        ppl = torch.exp(out["loss"])

    # Generate
    prompt = f"Instruction: {instruction}\nInput: {st}\nOutput:"
    prompt_enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_total_tokens-1)
    prompt_ids = prompt_enc["input_ids"].to(device)
    available = model.config.n_positions - prompt_ids.size(1)

    print(f"\n--- Example {idx} ---")
    print(f"  Tweet: {tweet[:90]}...")
    print(f"  Expected: {expected} | Loss: {out['loss'].item():.4f} | PPL: {ppl.item():.1f}")

    if available > 0:
        gen_ids = generate(model, prompt_ids, min(5, available), tokenizer.eos_token_id)
        gen_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        if "Output:" in gen_text:
            after = gen_text.split("Output:")[-1].strip()
            pred = after.split()[0] if after else "<empty>"
        else:
            pred = gen_text.strip().split()[0] if gen_text.strip() else "<empty>"
        match = "✓" if pred == expected else ""
        print(f"  Predicted: {pred} {match}")

Part 2 Test Set Benchmark (1000 examples)
  Avg Test Loss:       2.7451
  Avg Test Perplexity:  15.6

--- 5 Test Examples ---

--- Example 654 ---
  Tweet: Ordered a new computer! Unfortunately it won't get here until the end of June....
  Expected: sadness | Loss: 2.1934 | PPL: 9.0
  Predicted: worry 

--- Example 114 ---
  Tweet: ah thats better snow patrol! now to be stuck indoors  still weekend coming up and fresh ai...
  Expected: happiness | Loss: 3.9049 | PPL: 49.6
  Predicted: worry 

--- Example 25 ---
  Tweet: is at home with a pukey boy! Poor little baby...
  Expected: worry | Loss: 2.1392 | PPL: 8.5
  Predicted: worry ✓

--- Example 759 ---
  Tweet: urgh, i really hate that medicine...
  Expected: hate | Loss: 1.4476 | PPL: 4.3
  Predicted: worry 

--- Example 281 ---
  Tweet: @nickkk_ that sucks!...
  Expected: sadness | Loss: 1.0921 | PPL: 3.0
  Predicted: <empty> 


---

### 💾 Saving the Custom Model

The custom model can be saved as a standard PyTorch checkpoint:

```python
torch.save(model.state_dict(), "./models/gpt2-custom.pt")
```

Or converted to a Hugging Face-compatible format using `save_pretrained()` (requires wrapping in the HF model class, which is not covered here).

---

## 🎓 Summary & Key Takeaways

This tutorial walked through the complete lifecycle of **training a GPT-2-style LLM from random weights** — both with high-level APIs and from scratch. The models were evaluated on a proper **train/val/test split** with benchmark tables.

### What You Learned

| Concept | Part 1 (HF API) | Part 2 (From Scratch) |
|---------|-----------------|----------------------|
| **Model creation** | `GPT2Config` + `GPT2LMHeadModel` (random init) | Hand-coded `GPT2Attention`, `GPT2MLP`, `GPT2Block`, `GPT2Model`, `GPT2LMHeadModel` |
| **Tokenizer** | Pretrained GPT-2 BPE (ModelScope) | (Same tokenizer, reused) |
| **Data splits** | 5000 train / 1000 val / 1000 test | (Same splits, reused) |
| **Training** | `Trainer` API with val monitoring, fp16 | Manual loop with per-epoch val loss |
| **Inference** | `model.generate()` | Custom greedy `generate()` function |
| **Model size** | ~70M parameters | ~25M parameters |

### Key Architectural Insights

1. **Causal masking** ensures position $i$ can only attend to positions $\leq i$ — this is what makes the model autoregressive
2. **Pre-norm** (LayerNorm before each sublayer, not after) is more stable and standard in modern LLMs
3. **Weight tying** between `wte` and `lm_head` saves parameters and improves training
4. **Label shifting** (predict token $n+1$ from token $n$) is how causal LMs are trained — the model's own input becomes its target

### Why Train from Scratch?

> **🔑 This is native GPT-2 training — NOT fine-tuning.** The model starts with random weights and learns everything from the data. The only pretrained component we borrow is the **tokenizer** (BPE vocabulary and merge rules), because training a tokenizer from scratch requires a separate corpus and is a distinct skill. Think of it as re-running GPT-2's original training recipe on a small scale — the same principle that produced models like GPT-2, Llama, and DeepSeek, just on a much smaller dataset.

### Important Limitations

> **⚠️ This is a teaching exercise.** The model is trained on a small sentiment-classification dataset and will NOT behave like a general-purpose LLM. It learns to output sentiment labels, not to reason or converse.

To train a real LLM, you would need:
- A **much larger diverse corpus** (web text, books, code, etc.)
- **More parameters** (hundreds of millions to billions)
- **Longer training** (days to weeks on many GPUs)
- **Instruction tuning + RLHF** for helpful assistant behavior

---

---

## 📬 Contact

For questions, collaboration, or feedback on this tutorial series, please reach out:

- **Professional/HR:** `yucongcai_business@outlook.com`
- **Research:** `yucongcai_research@outlook.com`

---

## 📋 Version History

| Version | Date | Changes |
|---------|------|---------|
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/` (archive kept untouched) |
| v1.1 | 2026-08-04 | **Major documentation overhaul:** detailed markdown, docstrings, architecture diagram, fixed `max_total_tokens` consistency |
| v1.2 | 2026-08-04 | **Structural overhaul:** train/val/test splits (5K/1K/1K), val monitoring, benchmark tables, transformers 5.x compat fixes |
| v1.3 | 2026-08-04 | **Part 2 correctness fix:** GPT-2 standard init, embedding dropout, bias, proper padding masks, mode-collapse fix (label distribution diagnosis → 18M model) |

### v1.3 changes (2026-08-04)

| Change |
|--------|
| **Part 2 model capacity:** 7M → 18M (`n_embd=256, n_layer=6, n_head=8`) to fix mode collapse (all-output-"worry") |
| **Label distribution diagnosis:** Added cell showing 13-class imbalance, token budget breakdown, signal dilution |
| **Weight initialization:** PyTorch default Kaiming → GPT-2 standard `Normal(0, 0.02)` with `_init_weights()` |
| **Embedding dropout:** Added `self.drop` in `GPT2Model` (standard in GPT-2; was missing) |
| **Linear bias:** All `nn.Linear` now `bias=True` (QKV, MLP, output projection) |
| **LayerNorm eps:** Added `eps=1e-5` (matches GPT-2 standard) |
| **Generate attention mask:** Custom `generate()` now passes `torch.ones` mask (was calling model without mask) |
| **Training:** 10 epochs/lr=5e-4 → 15 epochs/lr=3e-4 for 18M model |
| **Data pipeline:** Part 2 has self-contained tokenization (rebuilds splits with same seed 42) |
| **Batch inspection:** Added shape/label/mask verification in Part 2 pipeline cell |
| **Token-ID padding visualisation:** Shows raw IDs, attention mask, real/pad counts per sample |
| **Part 1 unchanged:** All Part 1 cells remain as-is from v1.2 |